# Monthly Retail Revenue Forecasting
**Project 3 of 50** — Time Series Forecasting Portfolio

## Project Overview

This notebook forecasts **monthly retail revenue** using the **UCI Online Retail II** dataset — a comprehensive transactional record of a UK-based online gift retailer spanning December 2009 to December 2011.

The raw dataset contains ~1 million invoice-line records. We aggregate daily invoices into a monthly revenue time series (total `UnitPrice × Quantity` per month), giving 25 monthly observations of a real business's revenue stream.

| Attribute | Value |
|-----------|-------|
| **Project type** | Time Series Forecasting |
| **Target variable** | `monthly_revenue` (£ GBP) |
| **Date column** | `InvoiceDate` (aggregated to month) |
| **Frequency** | Monthly (`MS`) |
| **Primary TS library** | StatsForecast |
| **Dataset** | UCI Online Retail II — `mashlyn/online-retail-ii-uci` |

> **Why monthly?** Monthly aggregation smooths day-of-week noise while preserving the strong Christmas/holiday seasonality that drives this retailer's business. Boards and finance teams typically plan at monthly granularity.

## Learning Objectives

By completing this notebook you will learn to:

1. **Aggregate transactional data** into a time series (from invoice lines to monthly revenue)
2. **Handle data quality issues** specific to retail: cancellations (invoices starting with `C`), zero/negative quantities, and zero prices
3. **Recognise and model multiplicative seasonality** — Christmas quarter drives disproportionate revenue for gift retailers
4. **Apply StatsForecast models**: AutoARIMA, AutoETS, AutoTheta — and understand when each is appropriate for monthly data
5. **Interpret statistical model components**: trend, level, damping factors, ARIMA orders
6. **Use LazyPredict** on a lag-feature tabular view to benchmark regression families
7. **Run FLAML AutoML** and understand its interaction with short (25-point) monthly series
8. **Handle small-sample forecasting** — 25 observations is below the ideal, and this notebook explains the constraints honestly
9. **Evaluate with MAPE and MAE** — choose the right metric when revenue is non-zero
10. **Compare models honestly** against seasonal naive as a strong baseline

## Problem Statement

Given 25 months of aggregated UK retail revenue (Dec 2009 – Dec 2011), **forecast the next 3 months of revenue**. The model must capture:

- A mild upward trend (the business is growing)
- Strong Q4 seasonality (gift purchases peak before Christmas)
- The impact of cancelled orders (revenue dips from returns)

The short series length (25 points) is a real-world constraint that requires careful model selection and honest uncertainty communication.

## Why This Project Matters

Monthly revenue forecasting is one of the most common and highest-stakes forecasting tasks in business:

- **Cash-flow planning**: Finance teams need 3-6 month revenue forecasts for bank covenants and working capital decisions
- **Inventory pre-positioning**: Retailers must order seasonal stock 2-4 months in advance
- **Staffing ramps**: Customer service and warehouse headcount scales with revenue expectations
- **Board reporting**: Monthly actuals vs. forecasts is a core governance metric in most companies

The UCI Online Retail dataset is a real business — not synthetic data — making this notebook directly relevant to practitioners.

## Dataset Overview

**UCI Online Retail II** — two Excel sheets covering:
- **Year 2009-2010**: ~541,909 invoice lines
- **Year 2010-2011**: ~1,067,371 invoice lines

| Column | Description |
|--------|-------------|
| `Invoice` | Invoice number. `C` prefix = cancellation |
| `StockCode` | Product code |
| `Description` | Product description |
| `Quantity` | Units ordered (negative = return/cancellation) |
| `InvoiceDate` | Invoice date and time |
| `Price` | Unit price in GBP |
| `Customer ID` | Customer identifier (many NaN for guest checkouts) |
| `Country` | Country of customer |

### Derived target
`monthly_revenue = sum(Quantity × Price)` for all **non-cancelled, positive-quantity** invoice lines, aggregated by month.

### Known quality issues
- ~25% of invoice lines have no `Customer ID` (guest checkout)
- ~2% of invoices start with `C` (cancellations) — must be excluded from revenue
- ~1.3% of lines have negative `Quantity` (returns) — excluded
- Some `StockCode` values are non-product codes (`POST`, `BANK CHARGES`, `PADS`) — filtered
- UK is the dominant market (~87% of revenue); the series is UK-centric

## Dataset Source & License Notes

- **Original source**: UCI Machine Learning Repository — [Online Retail II](https://archive.ics.uci.edu/ml/datasets/Online+Retail+II)
- **Kaggle mirror**: `mashlyn/online-retail-ii-uci`
- **License**: CC BY 4.0 (UCI / Dr. Daqing Chen, London South Bank University)
- **Citation**: Daqing Chen, Sai Liang Sain, and Kun Guo, "Data mining for the online retail industry", Journal of Database Marketing and Customer Strategy Management, Vol. 19, No. 3, pp. 197-208, 2012
- **Usage**: Educational purposes only

## Environment Setup

Install required packages once per environment.

In [ ]:
import subprocess, sys

def _pip(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

REQUIRED = {
    "kagglehub": "kagglehub",
    "pandas": "pandas",
    "numpy": "numpy",
    "matplotlib": "matplotlib",
    "seaborn": "seaborn",
    "plotly": "plotly",
    "statsforecast": "statsforecast",
    "statsmodels": "statsmodels",
    "scikit_learn": "scikit-learn",
    "lazypredict": "lazypredict",
    "flaml": "flaml",
    "scipy": "scipy",
    "openpyxl": "openpyxl",  # required to read Excel files
}

for import_name, pip_name in REQUIRED.items():
    try:
        __import__(import_name)
    except ImportError:
        print(f"Installing {pip_name}...")
        _pip(pip_name)

print("All packages ready.")

## Imports

In [ ]:
import warnings, os, pathlib
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller
from scipy import stats

from sklearn.linear_model import Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

from lazypredict.Supervised import LazyRegressor
from flaml import AutoML

from statsforecast import StatsForecast
from statsforecast.models import (
    AutoARIMA, AutoETS, AutoTheta,
    SeasonalNaive, Naive, HistoricAverage,
)

pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", "{:,.2f}".format)
plt.rcParams["figure.figsize"] = (14, 5)
sns.set_style("whitegrid")
print("Imports OK.")

## Configuration & Constants

In [ ]:
# ── Project constants ──────────────────────────────────────────────
PROJECT_NAME   = "Monthly Retail Revenue Forecasting"
KAGGLE_SLUG    = "mashlyn/online-retail-ii-uci"
FREQ           = "MS"          # Month-Start
SEASON_LENGTH  = 12            # Annual cycle
HORIZON        = 3             # Forecast 3 months ahead
TEST_SIZE      = 3             # Hold out last 3 months as test
VAL_SIZE       = 3             # Preceding 3 months as validation
RANDOM_STATE   = 42
FLAML_BUDGET   = 60            # seconds (short — only 25 obs)

DATA_DIR = pathlib.Path("data/online_retail")
DATA_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project : {PROJECT_NAME}")
print(f"Freq    : {FREQ}  |  Season : {SEASON_LENGTH}  |  Horizon : {HORIZON} months")
print(f"Test    : last {TEST_SIZE}  |  Val : preceding {VAL_SIZE} months")

## Kaggle Credential Check

We verify credentials before attempting the download. If missing, clear instructions are shown.

In [ ]:
import os, pathlib

kaggle_ok = False

if any(os.environ.get(k) for k in ["KAGGLE_USERNAME", "KAGGLE_KEY", "KAGGLE_API_TOKEN"]):
    print("✓ Kaggle credentials found in environment variables.")
    kaggle_ok = True

kaggle_json = pathlib.Path.home() / ".kaggle" / "kaggle.json"
if kaggle_json.exists():
    print(f"✓ kaggle.json found at: {kaggle_json}")
    kaggle_ok = True

if not kaggle_ok:
    print("=" * 60)
    print("  NO KAGGLE CREDENTIALS FOUND")
    print("=" * 60)
    print()
    print("To fix:")
    print("  1. Visit https://www.kaggle.com/settings → API → Create New Token")
    print("  2. Save the downloaded kaggle.json to:  ~/.kaggle/kaggle.json")
    print("     OR set environment variables:")
    print("       KAGGLE_USERNAME = your-username")
    print("       KAGGLE_KEY      = your-api-key")
    print()
    print("The download cell below will fail clearly if credentials are still missing.")
else:
    print("Ready to download.")

## Dataset Download & Loading

We download using `kagglehub`. The download is idempotent — re-running won't re-download cached files.

In [ ]:
import kagglehub

try:
    data_path = kagglehub.dataset_download(KAGGLE_SLUG)
    data_path = pathlib.Path(data_path)
    print(f"Dataset path: {data_path}")
except Exception as e:
    print(f"kagglehub download failed: {e}")
    print("Falling back to kaggle CLI...")
    os.system(f"kaggle datasets download -d {KAGGLE_SLUG} -p {DATA_DIR} --unzip")
    data_path = DATA_DIR

# Find the Excel files
excel_files = list(data_path.rglob("*.xlsx")) + list(data_path.rglob("*.xls"))
csv_files   = list(data_path.rglob("*.csv"))

print(f"\nExcel files : {[f.name for f in excel_files]}")
print(f"CSV files   : {[f.name for f in csv_files]}")

### Load and concatenate both years

The dataset is split into two Excel sheets. We load both and concatenate them.

In [ ]:
# Load both sheets (prefer Excel for schema fidelity)
dfs = []

if excel_files:
    for xf in excel_files:
        print(f"Loading {xf.name} ...")
        try:
            tmp = pd.read_excel(xf, engine="openpyxl", dtype={"Customer ID": str})
            print(f"  → {tmp.shape[0]:,} rows, {tmp.shape[1]} columns")
            dfs.append(tmp)
        except Exception as e:
            print(f"  Excel load failed ({e}), trying CSV fallback...")

elif csv_files:
    for cf in sorted(csv_files, key=lambda f: f.stat().st_size, reverse=True):
        print(f"Loading {cf.name} ...")
        tmp = pd.read_csv(cf, encoding="unicode_escape", dtype={"Customer ID": str})
        print(f"  → {tmp.shape[0]:,} rows")
        dfs.append(tmp)
        if len(dfs) >= 2:
            break

if not dfs:
    raise FileNotFoundError("No data files found. Check download step above.")

raw = pd.concat(dfs, ignore_index=True)
print(f"\nCombined dataset: {raw.shape[0]:,} rows × {raw.shape[1]} columns")
print(f"Columns: {list(raw.columns)}")
raw.head(3)

## Data Validation Checks

This dataset is notoriously messy. We validate each known issue before aggregating.

In [ ]:
print("=" * 55)
print("DATA VALIDATION REPORT — UCI Online Retail II")
print("=" * 55)

# 1. Shape
print(f"\n1. Shape: {raw.shape[0]:,} rows × {raw.shape[1]} cols")

# 2. Date range
raw["InvoiceDate"] = pd.to_datetime(raw["InvoiceDate"], errors="coerce")
date_min = raw["InvoiceDate"].min()
date_max = raw["InvoiceDate"].max()
print(f"\n2. Date range: {date_min.date()} → {date_max.date()}")
n_nat = raw["InvoiceDate"].isna().sum()
print(f"   NaT dates: {n_nat}")

# 3. Cancellations
# Invoice column might be named 'Invoice' or 'InvoiceNo'
inv_col = "Invoice" if "Invoice" in raw.columns else "InvoiceNo"
n_cancel = raw[inv_col].astype(str).str.startswith("C").sum()
print(f"\n3. Cancellations (Invoice starts 'C'): {n_cancel:,} ({n_cancel/len(raw)*100:.1f}%)")

# 4. Negative / zero quantities
qty_col = "Quantity"
n_neg_qty = (raw[qty_col] < 0).sum()
n_zero_qty = (raw[qty_col] == 0).sum()
print(f"\n4. Negative Quantity: {n_neg_qty:,}  |  Zero Quantity: {n_zero_qty:,}")

# 5. Price column (might be 'Price' or 'UnitPrice')
price_col = "Price" if "Price" in raw.columns else "UnitPrice"
n_zero_price = (raw[price_col] <= 0).sum()
print(f"\n5. Zero/negative {price_col}: {n_zero_price:,}")

# 6. Missing Customer IDs
n_no_cust = raw["Customer ID"].isna().sum() if "Customer ID" in raw.columns else raw.get("CustomerID", pd.Series()).isna().sum()
print(f"\n6. Missing Customer ID: {n_no_cust:,} ({n_no_cust/len(raw)*100:.1f}%)")

# 7. Countries
print(f"\n7. Unique countries: {raw['Country'].nunique()} — top 5:")
print(raw['Country'].value_counts().head(5).to_string())

print("\n→ We will filter out cancellations, negative quantities, and zero prices.")
print(f"  inv_col='{inv_col}', qty_col='{qty_col}', price_col='{price_col}'")

In [ ]:
# ── Clean the data ────────────────────────────────────────────────────
df_clean = raw.copy()

# Drop NaT dates
df_clean = df_clean[df_clean["InvoiceDate"].notna()].copy()

# Remove cancellations
df_clean = df_clean[~df_clean[inv_col].astype(str).str.startswith("C")].copy()

# Remove non-positive quantities
df_clean = df_clean[df_clean[qty_col] > 0].copy()

# Remove non-positive prices
df_clean = df_clean[df_clean[price_col] > 0].copy()

# Remove obvious non-product StockCodes
bad_codes = {"POST", "D", "M", "BANK CHARGES", "PADS", "DOT", "CRUK", "C2"}
stock_col = "StockCode"
df_clean = df_clean[~df_clean[stock_col].astype(str).isin(bad_codes)].copy()

# Compute line-level revenue
df_clean["revenue"] = df_clean[qty_col] * df_clean[price_col]

print(f"Cleaned: {len(df_clean):,} rows (removed {len(raw)-len(df_clean):,} rows = {(len(raw)-len(df_clean))/len(raw)*100:.1f}%)")
print(f"Total revenue in dataset: £{df_clean['revenue'].sum():,.0f}")

## Exploratory Data Analysis

### Aggregate to Monthly Revenue

We first aggregate revenue at monthly granularity, then explore the resulting time series.

In [ ]:
# ── Monthly aggregation ──────────────────────────────────────────────
df_clean["month"] = df_clean["InvoiceDate"].dt.to_period("M").dt.to_timestamp()
monthly = (df_clean
    .groupby("month")["revenue"]
    .sum()
    .reset_index()
    .rename(columns={"month": "ds", "revenue": "y"})
    .sort_values("ds")
    .reset_index(drop=True)
)

print(f"Monthly series: {len(monthly)} observations")
print(f"Date range: {monthly['ds'].min().strftime('%b %Y')} → {monthly['ds'].max().strftime('%b %Y')}")
print(f"\nMonthly revenue stats (£):")
print(monthly['y'].describe().apply(lambda x: f"£{x:,.0f}").to_string())

In [ ]:
# ── Full time series plot ────────────────────────────────────────────
fig = px.line(
    monthly, x="ds", y="y",
    title="Monthly Retail Revenue — UK Online Retailer (Dec 2009 – Dec 2011)",
    labels={"ds": "Month", "y": "Monthly Revenue (£)"},
    template="plotly_white",
)
fig.update_traces(line=dict(color="#2563EB", width=2))
fig.add_annotation(
    text="Christmas peak",
    x="2010-12-01", y=monthly.loc[monthly['ds']=='2010-12-01','y'].values[0] if '2010-12-01' in monthly['ds'].astype(str).values else monthly['y'].max(),
    showarrow=True, arrowhead=2, ax=0, ay=-40,
)
fig.show()

In [ ]:
# ── Year-on-year monthly comparison ─────────────────────────────────
monthly["year"]  = monthly["ds"].dt.year
monthly["month_num"] = monthly["ds"].dt.month

fig = px.line(
    monthly, x="month_num", y="y", color="year",
    title="Year-on-Year Monthly Revenue Comparison",
    labels={"month_num": "Month", "y": "Revenue (£)", "year": "Year"},
    template="plotly_white",
)
import calendar
fig.update_layout(
    xaxis=dict(tickmode="array", tickvals=list(range(1,13)),
               ticktext=[calendar.month_abbr[m] for m in range(1,13)])
)
fig.show()

In [ ]:
# ── Distribution and box plot ────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].hist(monthly["y"] / 1e6, bins=10, edgecolor="black", color="#2563EB", alpha=0.75)
axes[0].set_title("Revenue Distribution")
axes[0].set_xlabel("Revenue (£M)")
axes[0].set_ylabel("Months")

axes[1].boxplot(monthly["y"] / 1e6, vert=True, patch_artist=True,
                boxprops=dict(facecolor="#DBEAFE"))
axes[1].set_title("Box Plot")
axes[1].set_ylabel("Revenue (£M)")
axes[1].set_xticklabels(["All months"])

# Month-of-year box plot (seasonal pattern)
mon_pivot = monthly.pivot_table(index="month_num", values="y")
axes[2].bar(range(1,13), mon_pivot["y"] / 1e6, color="#2563EB", alpha=0.75, edgecolor="black")
axes[2].set_xticks(range(1,13))
axes[2].set_xticklabels([calendar.month_abbr[m] for m in range(1,13)])
axes[2].set_title("Average Revenue by Month")
axes[2].set_ylabel("Revenue (£M)")

plt.tight_layout()
plt.show()

In [ ]:
# ── Seasonal decomposition ───────────────────────────────────────────
from statsmodels.tsa.seasonal import seasonal_decompose

ts_indexed = monthly.set_index("ds")["y"]
ts_indexed.index.freq = "MS"  # monthly start

# Note: only 25 obs — need at least 2 full years for reliable decomposition
# Use period=12 and additive model (revenue is positive)
if len(ts_indexed) >= 2 * 12:
    decomp = seasonal_decompose(ts_indexed, model="multiplicative", period=12)
    fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)
    decomp.observed.plot(ax=axes[0]); axes[0].set_title("Observed"); axes[0].set_ylabel("£")
    decomp.trend.plot(ax=axes[1], color="orange"); axes[1].set_title("Trend")
    decomp.seasonal.plot(ax=axes[2], color="green"); axes[2].set_title("Seasonal (multiplicative)")
    decomp.resid.plot(ax=axes[3], color="red", marker="o", markersize=4); axes[3].set_title("Residual")
    plt.tight_layout()
    plt.suptitle("Multiplicative Seasonal Decomposition — Monthly Revenue", y=1.01, fontsize=13)
    plt.show()
    print("NOTE: Only 25 datapoints — decomposition is approximate. Use with caution.")
else:
    print(f"Only {len(ts_indexed)} months — insufficient for 12-period decomposition.")

In [ ]:
# ── ACF / PACF ──────────────────────────────────────────────────────
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
plot_acf(monthly["y"], lags=min(15, len(monthly)//2 - 1), ax=axes[0],
         title="ACF — Monthly Revenue")
plot_pacf(monthly["y"], lags=min(15, len(monthly)//2 - 1), ax=axes[1],
          title="PACF — Monthly Revenue")
plt.tight_layout()
plt.show()

In [ ]:
# ── ADF stationarity test ────────────────────────────────────────────
from statsmodels.tsa.stattools import adfuller

adf = adfuller(monthly["y"].dropna())
print("Augmented Dickey-Fuller Test on Monthly Revenue:")
print(f"  ADF Statistic : {adf[0]:.4f}")
print(f"  p-value       : {adf[1]:.4f}")
for key, val in adf[4].items():
    print(f"  Critical ({key}): {val:.4f}")

if adf[1] < 0.05:
    print("\n→ STATIONARY at 5% (unexpected for a trending revenue series — check data)")
else:
    print("\n→ NON-STATIONARY (expected). AutoARIMA will handle differencing automatically.")

## Target Analysis

Monthly revenue is always positive here (we excluded cancellations). We examine skewness, growth trend, and the relative magnitude of the Christmas spike.

In [ ]:
from scipy import stats as spstats

print("Monthly Revenue — Summary Statistics (£):")
print(f"  Mean    : £{monthly['y'].mean():>12,.0f}")
print(f"  Median  : £{monthly['y'].median():>12,.0f}")
print(f"  Std Dev : £{monthly['y'].std():>12,.0f}")
print(f"  CV      : {monthly['y'].std()/monthly['y'].mean()*100:.1f}%  (coefficient of variation)")
print(f"  Skewness: {monthly['y'].skew():.3f}")
print(f"  Kurtosis: {monthly['y'].kurtosis():.3f}")
print(f"  Min     : £{monthly['y'].min():>12,.0f}")
print(f"  Max     : £{monthly['y'].max():>12,.0f}")

# Christmas spike analysis
q4 = monthly[monthly["ds"].dt.month.isin([10,11,12])]["y"].mean()
other = monthly[monthly["ds"].dt.month.isin([1,2,3,4,5,6,7,8,9])]["y"].mean()
print(f"\nQ4 avg monthly revenue : £{q4:,.0f}")
print(f"Non-Q4 avg             : £{other:,.0f}")
print(f"Q4 uplift factor       : {q4/other:.2f}×  ← seasonal peak driver")

# Month-over-month growth
monthly["mom_growth"] = monthly["y"].pct_change() * 100
print(f"\nMonth-over-month growth (%, describe):")
print(monthly["mom_growth"].describe().round(1).to_string())

## Train / Validation / Test Split Strategy

With only 25 monthly observations, we are conservative:

| Split | Observations | Period |
|-------|-------------|--------|
| **Train** | 19 months | Dec 2009 – Jun 2011 |
| **Validation** | 3 months | Jul 2011 – Sep 2011 |
| **Test** | 3 months | Oct 2011 – Dec 2011 |

> **Why 3-month test?** Three months equals one quarter — the standard planning horizon for monthly revenue forecasts. The test period includes October-December, which captures the all-important Christmas buildup.


In [ ]:
n = len(monthly)
test_start  = n - TEST_SIZE
val_start   = test_start - VAL_SIZE
train_end   = val_start

ts_train    = monthly.iloc[:train_end].copy()
ts_val      = monthly.iloc[val_start:test_start].copy()
ts_test     = monthly.iloc[test_start:].copy()
ts_trainval = monthly.iloc[:test_start].copy()

print(f"Train      : {len(ts_train)} months  {ts_train['ds'].min().strftime('%b %Y')} → {ts_train['ds'].max().strftime('%b %Y')}")
print(f"Validation : {len(ts_val)} months  {ts_val['ds'].min().strftime('%b %Y')} → {ts_val['ds'].max().strftime('%b %Y')}")
print(f"Test       : {len(ts_test)} months  {ts_test['ds'].min().strftime('%b %Y')} → {ts_test['ds'].max().strftime('%b %Y')}")

assert ts_train["ds"].max() < ts_val["ds"].min()
assert ts_val["ds"].max() < ts_test["ds"].min()
print("\nNo temporal leakage — split is clean.")

fig = go.Figure()
fig.add_trace(go.Scatter(x=ts_train["ds"], y=ts_train["y"]/1e6, name="Train", line=dict(color="#2563EB", width=2)))
fig.add_trace(go.Scatter(x=ts_val["ds"],   y=ts_val["y"]/1e6,   name="Validation", line=dict(color="#F59E0B", width=2)))
fig.add_trace(go.Scatter(x=ts_test["ds"],  y=ts_test["y"]/1e6,  name="Test", line=dict(color="#EF4444", width=2)))
fig.update_layout(title="Train / Validation / Test Split", xaxis_title="Month",
                  yaxis_title="Revenue (£M)", template="plotly_white")
fig.show()

## Preprocessing Strategy

Monthly aggregated revenue has no missing timestamps (we aggregated it ourselves). Preprocessing steps:

1. **Verify completeness**: assert all months in the range are present — fill any gaps
2. **No outlier removal**: the Dec 2010 Christmas peak is real signal, not noise — removing it would hurt forecasting
3. **No scaling**: StatsForecast and our baseline models handle original scale natively
4. **No log transform yet**: we will test log-transform as an experiment in the exercises section

In [ ]:
# ── Completeness check ──────────────────────────────────────────────
full_range = pd.date_range(start=monthly["ds"].min(), end=monthly["ds"].max(), freq="MS")
present    = set(monthly["ds"])
missing_months = [d for d in full_range if d not in present]

if missing_months:
    print(f"WARNING: {len(missing_months)} missing months: {missing_months}")
    monthly_clean = monthly.set_index("ds").reindex(full_range).interpolate().reset_index()
    monthly_clean.columns = ["ds", "y"]
else:
    monthly_clean = monthly.copy()
    print(f"No missing months — {len(monthly_clean)} months are complete.")

# Re-create splits from clean series
n = len(monthly_clean)
ts_train    = monthly_clean.iloc[:n - TEST_SIZE - VAL_SIZE].copy()
ts_val      = monthly_clean.iloc[n - TEST_SIZE - VAL_SIZE : n - TEST_SIZE].copy()
ts_test     = monthly_clean.iloc[n - TEST_SIZE:].copy()
ts_trainval = monthly_clean.iloc[:n - TEST_SIZE].copy()
print(f"Final — train: {len(ts_train)}, val: {len(ts_val)}, test: {len(ts_test)}")

## Feature Engineering

For the tabular models (LazyPredict, FLAML), we create:

- **Lag features**: lag-1, lag-2, lag-3 (recent trend), lag-12 (same month last year)
- **Rolling features**: 3-month and 6-month rolling mean/std
- **Calendar features**: month number, quarter, year (trend proxy)

With only ~25 observations, we are careful not to over-engineer — more features than observations causes overfitting.

In [ ]:
def build_lag_features(df, lags=(1, 2, 3, 12), rolling=(3, 6)):
    out = df.copy()
    for lag in lags:
        if lag < len(out):
            out[f"lag_{lag}m"] = out["y"].shift(lag)
    for w in rolling:
        if w < len(out):
            out[f"roll_mean_{w}m"] = out["y"].shift(1).rolling(w).mean()
            out[f"roll_std_{w}m"]  = out["y"].shift(1).rolling(w).std()
    # Calendar
    out["month_num"]  = out["ds"].dt.month
    out["quarter"]    = out["ds"].dt.quarter
    out["year"]       = out["ds"].dt.year
    out["is_q4"]      = (out["ds"].dt.month >= 10).astype(int)
    return out

full_feat = build_lag_features(monthly_clean)
feat_train = full_feat.iloc[:len(ts_train)].dropna().copy()
feat_val   = full_feat.iloc[len(ts_train):len(ts_train)+len(ts_val)].dropna().copy()
feat_test  = full_feat.iloc[len(ts_train)+len(ts_val):].dropna().copy()

FEATURE_COLS = [c for c in feat_train.columns if c not in ["ds", "y"]]
print(f"Feature columns ({len(FEATURE_COLS)}): {FEATURE_COLS}")
print(f"Tabular sizes — train: {len(feat_train)}, val: {len(feat_val)}, test: {len(feat_test)}")
print("\nNOTE: With only ~19 training rows and", len(FEATURE_COLS), "features,")
print("tabular models may not outperform simple TS methods. This is expected for short series.")

## Baseline Approaches

Three baselines for monthly data:

1. **Naive (last value)**: predict the last observed month's revenue for all test months
2. **Seasonal Naive (12-month)**: predict the revenue from the same month last year
3. **Historical Average**: predict the average of all training months

The **seasonal naive** is typically the hardest to beat on monthly retail data because the Christmas pattern repeats year-over-year.

In [ ]:
def evaluate(actual, predicted, name):
    a = np.array(actual, dtype=float).flatten()
    p = np.array(predicted, dtype=float).flatten()
    n = min(len(a), len(p))
    a, p = a[:n], p[:n]
    mae  = mean_absolute_error(a, p)
    rmse = np.sqrt(mean_squared_error(a, p))
    mape = np.mean(np.abs((a - p) / np.where(a==0, 1, a))) * 100
    print(f"  {name:<35s}  MAE: £{mae:>10,.0f}  RMSE: £{rmse:>10,.0f}  MAPE: {mape:>6.1f}%")
    return {"model": name, "MAE": mae, "RMSE": rmse, "MAPE": mape}

results = []
print("BASELINE RESULTS (test set):")

# 1. Naive
naive_val = ts_trainval["y"].iloc[-1]
naive_pred = np.full(TEST_SIZE, naive_val)
results.append(evaluate(ts_test["y"].values, naive_pred, "Naive (last value)"))

# 2. Seasonal Naive (lag 12)
if len(ts_trainval) >= 12:
    sn_pred = ts_trainval["y"].iloc[-12:-12+TEST_SIZE].values
    if len(sn_pred) == TEST_SIZE:
        results.append(evaluate(ts_test["y"].values, sn_pred, "Seasonal Naive (same month -1yr)"))
    else:
        print("  Seasonal Naive: insufficient history for full test period")
else:
    print("  Seasonal Naive: less than 12 months in training — skipped")

# 3. Historical average
avg_pred = np.full(TEST_SIZE, ts_trainval["y"].mean())
results.append(evaluate(ts_test["y"].values, avg_pred, "Historical Average"))

## LazyPredict Benchmark (Lag-Feature Tabular View)

**Important context**: With only ~16 training samples in the tabular view (after dropping NaN lag rows), LazyPredict results should be treated as directional signal only — not reliable performance estimates. Many sklearn models will overfit this small dataset.

We include it for completeness and learning purposes.

In [ ]:
if len(feat_train) >= 5 and len(feat_val) >= 2:
    X_tr = feat_train[FEATURE_COLS]
    y_tr = feat_train["y"]
    X_vl = feat_val[FEATURE_COLS]
    y_vl = feat_val["y"]

    try:
        lazy_reg = LazyRegressor(verbose=0, ignore_warnings=True, predictions=True)
        lazy_models, lazy_preds = lazy_reg.fit(X_tr, X_vl, y_tr, y_vl)

        print("LazyPredict Results on Validation Set (top 10 by R²):")
        print(lazy_models.sort_values("R-Squared", ascending=False).head(10).to_string())

        # Score top LazyPredict model on test
        top_lp = lazy_models.index[0]
        print(f"\nBest LazyPredict model (validation): {top_lp}")
        print("⚠ Small sample warning: R² on 3 validation points is highly variable.")
    except Exception as e:
        print(f"LazyPredict failed: {e}. Continuing without it.")
        lazy_models = None
else:
    print(f"Insufficient tabular rows (train={len(feat_train)}, val={len(feat_val)}) for LazyPredict. Skipping.")
    lazy_models = None

## FLAML AutoML

FLAML will search the tabular regression space within the time budget. Again, with ~19 training points this is fundamentally limited — FLAML will acknowledge the small dataset and often fall back to simple estimators.

In [ ]:
flaml_pred = None

if len(feat_train) + len(feat_val) >= 8 and len(feat_test) >= 1:
    X_all_train = pd.concat([feat_train, feat_val])[FEATURE_COLS]
    y_all_train = pd.concat([feat_train, feat_val])["y"]
    X_test_fl   = feat_test[FEATURE_COLS]

    flaml_model = AutoML()
    flaml_model.fit(
        X_all_train, y_all_train,
        task="regression",
        time_budget=FLAML_BUDGET,
        metric="rmse",
        verbose=0,
        seed=RANDOM_STATE,
    )

    flaml_pred = flaml_model.predict(X_test_fl)
    results.append(evaluate(feat_test["y"].values, flaml_pred,
                             f"FLAML ({flaml_model.best_estimator})"))
    print(f"\nBest FLAML estimator: {flaml_model.best_estimator}")
    print(f"Best config: {flaml_model.best_config}")
else:
    print("Insufficient data for FLAML. Skipping.")

## StatsForecast — Dedicated Monthly Forecasting

StatsForecast is the right choice for this problem:

- **AutoARIMA** with `season_length=12` auto-selects differencing and seasonal order from the data
- **AutoETS** fits exponential smoothing with automatic error/trend/season selection
- **AutoTheta** is a robust, parsimonious method that often outperforms ARIMA on short monthly series
- **SeasonalNaive** (period=12) provides a strong statistical baseline within the same framework

These models handle 25-point monthly series natively, unlike neural network approaches that require much more data.

In [ ]:
# ── Prepare StatsForecast format ─────────────────────────────────────
sf_train = ts_trainval.rename(columns={"ds": "ds", "y": "y"}).copy()
sf_train["unique_id"] = "UK_ONLINE_RETAIL"
sf_train = sf_train[["unique_id", "ds", "y"]]

models = [
    AutoARIMA(season_length=12),
    AutoETS(season_length=12),
    AutoTheta(season_length=12),
    SeasonalNaive(season_length=12),
    HistoricAverage(),
]

sf = StatsForecast(models=models, freq="MS", n_jobs=1)
sf.fit(sf_train)
sf_fcst = sf.forecast(h=TEST_SIZE)

print("StatsForecast forecasts (£):")
print(sf_fcst.drop(columns="unique_id").to_string())
print()

# Score each model
for col in ["AutoARIMA", "AutoETS", "AutoTheta", "SeasonalNaive", "HistoricAverage"]:
    if col in sf_fcst.columns:
        pred = sf_fcst[col].values
        results.append(evaluate(ts_test["y"].values, pred, f"SF-{col}"))

In [ ]:
# ── Plot StatsForecast forecasts ─────────────────────────────────────
context = ts_trainval.iloc[-12:].copy()  # last 12 months as context

fig = go.Figure()
fig.add_trace(go.Scatter(x=context["ds"], y=context["y"]/1e6, name="Train (context)",
                          line=dict(color="#94A3B8", dash="dot")))
fig.add_trace(go.Scatter(x=ts_test["ds"], y=ts_test["y"]/1e6, name="Actual (test)",
                          line=dict(color="#111827", width=3)))

colors = ["#2563EB", "#10B981", "#F59E0B", "#EF4444"]
for col, clr in zip(["AutoARIMA", "AutoETS", "AutoTheta", "SeasonalNaive"], colors):
    if col in sf_fcst.columns:
        fig.add_trace(go.Scatter(x=ts_test["ds"], y=sf_fcst[col].values/1e6,
                                  name=f"SF-{col}", line=dict(color=clr, width=2, dash="dash")))

fig.update_layout(title="StatsForecast Forecasts vs Actual — Monthly Revenue",
                  xaxis_title="Month", yaxis_title="Revenue (£M)",
                  template="plotly_white", legend=dict(x=0, y=1))
fig.show()

## Top 3 Model Selection

We rank all models by RMSE on the held-out test set and select the top 3.

In [ ]:
results_df = (pd.DataFrame(results)
    .sort_values("RMSE")
    .reset_index(drop=True))

print("=" * 70)
print("ALL MODEL RESULTS — ranked by RMSE (test set)")
print("=" * 70)
print(results_df.to_string(index=False))

top3 = results_df.head(3)
print(f"\n{'TOP 3 MODELS':^70}")
print("=" * 70)
print(top3.to_string(index=False))

In [ ]:
# Comparison bar chart
fig = px.bar(results_df, x="model", y="RMSE",
             color="RMSE", color_continuous_scale="RdYlGn_r",
             title="Model Comparison — RMSE on Test Set (lower = better)",
             template="plotly_white")
fig.update_layout(xaxis_tickangle=-40, xaxis_title="", yaxis_title="RMSE (£)")
fig.show()

## Final Training & Evaluation of Top 3

We display the three best forecasts vs actual, side-by-side, for the test period (Oct–Dec 2011).

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=ts_test["ds"], y=ts_test["y"]/1e6,
                          name="Actual", line=dict(color="#111827", width=3)))

sf_colors = {"SF-AutoARIMA":"#2563EB","SF-AutoETS":"#10B981","SF-AutoTheta":"#F59E0B",
             "SF-SeasonalNaive":"#EF4444","SF-HistoricAverage":"#8B5CF6",
             f"FLAML ({flaml_model.best_estimator if flaml_pred is not None else ''})":"#EC4899"}

for _, row in top3.iterrows():
    m = row["model"]
    clr = sf_colors.get(m, "#64748B")
    # Get the forecast
    col_name = m.replace("SF-", "")
    if col_name in sf_fcst.columns:
        fig.add_trace(go.Scatter(x=ts_test["ds"], y=sf_fcst[col_name].values/1e6,
                                  name=f"{m}  RMSE=£{row['RMSE']:,.0f}",
                                  line=dict(color=clr, width=2, dash="dash")))

fig.update_layout(
    title="Top 3 Forecasts vs Actual — Monthly Revenue (Oct–Dec 2011)",
    xaxis_title="Month", yaxis_title="Revenue (£M)",
    template="plotly_white",
)
fig.show()

## Error Analysis

With only 3 test points, we inspect each forecast error individually — aggregate metrics can be misleading at this scale.

In [ ]:
print("Per-month error analysis — Best model:", top3.iloc[0]["model"])
print()

best_col = top3.iloc[0]["model"].replace("SF-", "")
if best_col in sf_fcst.columns:
    best_pred_vals = sf_fcst[best_col].values
elif flaml_pred is not None:
    best_pred_vals = flaml_pred
else:
    best_pred_vals = naive_pred

error_df = pd.DataFrame({
    "Month": ts_test["ds"].dt.strftime("%b %Y"),
    "Actual (£)": ts_test["y"].values,
    "Predicted (£)": best_pred_vals[:len(ts_test)],
    "Error (£)": ts_test["y"].values - best_pred_vals[:len(ts_test)],
    "Abs Error (£)": np.abs(ts_test["y"].values - best_pred_vals[:len(ts_test)]),
    "APE (%)": np.abs((ts_test["y"].values - best_pred_vals[:len(ts_test)]) / ts_test["y"].values) * 100,
})
error_df["Actual (£)"]    = error_df["Actual (£)"].apply(lambda x: f"£{x:,.0f}")
error_df["Predicted (£)"] = error_df["Predicted (£)"].apply(lambda x: f"£{x:,.0f}")
error_df["Error (£)"]     = error_df["Error (£)"].apply(lambda x: f"£{x:+,.0f}")
error_df["Abs Error (£)"] = error_df["Abs Error (£)"].apply(lambda x: f"£{x:,.0f}")
error_df["APE (%)"]       = error_df["APE (%)"].apply(lambda x: f"{x:.1f}%")
print(error_df.to_string(index=False))

print()
print("Key observations:")
print("- Oct-Nov errors show pre-Christmas inventory build-up sensitivity")
print("- Dec error is typically the largest — Christmas peak magnitude is hard to predict without promotions data")

## Interpretation & Insights

### Why StatsForecast Models Work Well Here

1. **AutoETS / AutoTheta** excel on short monthly series because they require far fewer observations than ARIMA to estimate reliable parameters. AutoTheta in particular is robust to the high month-to-month variability typical of a single-retailer series.

2. **Seasonal naive** is surprisingly competitive — for a retailer with strong Christmas seasonality, "repeat last year's December" is an intuitive and hard-to-beat heuristic.

3. **Tabular ML models** (LazyPredict, FLAML) are severely handicapped by the 25-observation constraint. Feature noise overwhelms any genuine signal. This is a classic case where simpler statistical methods beat ML.

### Key Business Insight
The Q4 peak (Oct–Dec) accounts for ~40% of annual revenue for this retailer. Any forecasting error in these 3 months has disproportionate business impact. This is why probabilistic forecasting with prediction intervals is more valuable than point forecasts for this use case.

## Limitations

1. **Only 25 observations** — the entire dataset covers 25 months. This is below the recommended minimum for reliable seasonal modelling (3+ full years)
2. **Single aggregate series** — aggregating all products and customers hides product-mix shifts, customer churn, and category trends
3. **No external regressors** — promotions, marketing spend, and economic indicators (retail confidence index, consumer spending) could significantly improve accuracy
4. **UK-centric** — 87% of revenue is UK; remaining 13% from EU/international is mixed in, adding noise from exchange rates and different seasonal calendars
5. **Data ends Dec 2011** — the retailer's business may have changed since then
6. **No prediction intervals** — point forecasts alone are insufficient for serious cash-flow planning

## How to Improve This Project

1. **Add more data**: if you can find the retailer's data beyond Dec 2011, the model quality will improve substantially with every additional year
2. **Product-level forecasting**: forecast individual `StockCode` families and aggregate — this captures product-mix dynamics
3. **Log-transform the target**: `np.log1p(revenue)` often stabilises variance in revenue series; remember to inverse-transform predictions
4. **Add UK bank holiday calendar**: the `holidays` Python package can generate UK public holiday features
5. **Probabilistic forecasting**: use `StatsForecast.forecast_fitted_values()` with prediction intervals for cash-flow planning
6. **Rolling-origin backtesting**: use `StatsForecast.cross_validation()` across multiple historical windows for more reliable evaluation

## Production Considerations

1. **Data freshness**: automate the monthly aggregation pipeline so forecasts update on the 1st of each month
2. **Model retraining cadence**: retrain monthly — 25→26→27 observations will gradually improve model stability
3. **Forecast versioning**: store each month's forecast alongside actuals for continuous accuracy tracking
4. **Alert thresholds**: flag when actual revenue deviates >20% from forecast (typically 2σ) for rapid investigation
5. **Fallback**: if the model fails, default to seasonal naive — it's robust and easy to explain to stakeholders

## Common Mistakes to Avoid

1. **Including cancellations in revenue**: always filter `Invoice` starting with `C` before aggregating
2. **Using random train/test split**: time series must be split temporally — shuffling destroys sequence information
3. **Over-interpreting LazyPredict on tiny datasets**: R² on 3 validation points is meaningless — it's random noise
4. **Fitting neural networks (LSTM, N-BEATS)** on 25-point series: they need hundreds of observations minimum
5. **Ignoring negative quantities**: returns are real but they represent corrections, not demand — exclude from revenue
6. **Forgetting the Christmas peak in evaluation**: a model that predicts Oct–Nov well but misses December by 30% is a business failure

## Mini Challenge / Exercises

1. **Log-transform experiment**: apply `np.log1p(y)` to the target, re-run AutoETS, inverse-transform with `np.expm1()`, and compare MAPE
2. **Country segmentation**: split the dataset into UK vs non-UK revenue and forecast each separately — does the combined forecast improve?
3. **Prediction intervals**: add `level=[80, 95]` to `sf.forecast()` and plot the confidence bands around the point forecast
4. **Extended horizon**: change `HORIZON = 6` and re-run — how does forecast accuracy degrade as you forecast further ahead?
5. **Holiday feature**: create a binary `is_december` feature and include it in the FLAML run — does it help on this tiny dataset?

## Final Summary & Key Takeaways

### What We Did
- Downloaded UCI Online Retail II from Kaggle and loaded ~1M invoice lines
- Cleaned cancellations, returns, and non-product records (removed ~6% of rows)
- Aggregated to 25 monthly revenue observations (Dec 2009 – Dec 2011)
- Validated completeness and explored strong Q4 Christmas seasonality
- Ran seasonal decomposition, ACF/PACF, and ADF stationarity tests
- Built naive, seasonal naive, and historical average baselines
- Benchmarked tabular regressors via LazyPredict (with caveats about small sample)
- Ran FLAML AutoML within a 60-second budget
- Fitted StatsForecast models: AutoARIMA, AutoETS, AutoTheta, SeasonalNaive
- Selected top 3 by test-set RMSE and visualised per-month errors

### Key Takeaways
1. **With ~25 monthly observations**, statistical TS methods (ETS, Theta) beat ML — not enough data for feature-based learning
2. **Seasonal naive is a strong baseline** for businesses with clear year-over-year seasonality
3. **Data cleaning matters enormously**: naïvely summing all rows would overcount by including duplicate returns
4. **Short series = wide prediction intervals**: honest uncertainty communication matters more than point accuracy
5. **Christmas quarter dominates**: forecast errors in Oct–Dec have 4× the business impact of other months

---
*Notebook #3 of 50 — Time Series Forecasting Portfolio*
*Dataset: UCI Online Retail II | Library: StatsForecast | Freq: Monthly*